In [0]:
from pyspark.sql.functions import explode, col

df = spark.read.option("multiLine", "true").json("/FileStore/tables/brzydki.json")

df_flat = df.select(explode("features").alias("feature"))

df_selected = df_flat.select(
    col("feature.properties.featureId").alias("featureId"),
    col("feature.properties.toid").alias("toid"),
    col("feature.properties.changeEventType").alias("changeEventType"),
    col("feature.properties.jobReference").alias("jobReference"),
    col("feature.properties.validFromTimestamp").alias("validFrom"),
    col("feature.geometry.type").alias("geometryType"),
    col("feature.geometry.coordinates").alias("coordinates"),
    col("feature.properties.baseFormComponent.form").alias("baseForm"),
    col("feature.properties.lifecycleStatusComponent.lifecycleStatus").alias("lifecycleStatus"),
    col("feature.properties.baseFunctionComponents")[0]["function"].alias("function")
)


df_selected.show(truncate=False)


+------------------------------------+--------------------+---------------+------------+--------------------+------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------+------------------+---------------+-----------------+
|featureId                           |toid                |changeEventType|jobReference|validFrom           |geometryType|coordinates                                                                                                                                                |baseForm          |lifecycleStatus|function         |
+------------------------------------+--------------------+---------------+------------+--------------------+------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------+------------------+---------------+-----------------+
|a6a

## 1. ensure_required_columns(df, required_columns)

Cel: Zapobiega przypadkowemu brakowi kluczowych kolumn w danych wejściowych.


In [0]:
def ensure_required_columns(df, required_columns: list):
    """
    Validates that all required columns are present in the DataFrame.

    :param df: pd.DataFrame or pyspark.sql.DataFrame
    :param required_columns: List of expected column names
    """
    actual_columns = df.columns
    missing = [col for col in required_columns if col not in actual_columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")


## 2. assert_unique_keys(df, key_columns)
Cel: Chroni przed duplikatami w danych, które mają być jednoznacznie identyfikowalne.

In [0]:

def assert_unique_keys(df, key_columns: list):
    """
    Ensures that the combination of key_columns is unique.

    :param df: pd.DataFrame
    :param key_columns: List of column names that form the unique key
    """
    if df.duplicated(subset=key_columns).any():
        raise ValueError(f"Duplicate records found for keys: {key_columns}")



## 3. validate_value_ranges(df, column_ranges)
Cel: Chroni przed błędnymi wartościami liczbowymi (np. ujemny wiek, zbyt duża liczba itp.)

In [0]:
def validate_value_ranges(df, column_ranges: dict):
    """
    Checks if numerical values in columns fall within expected ranges.

    :param df: pd.DataFrame
    :param column_ranges: Dict of {column_name: (min, max)}
    """
    for col, (min_val, max_val) in column_ranges.items():
        if col in df.columns:
            if not df[col].between(min_val, max_val).all():
                raise ValueError(f"Column '{col}' has values outside range {min_val}-{max_val}")


## 4. auto_cast_columns(df, schema)
Cel: Automatyczne rzutowanie kolumn na oczekiwane typy z walidacją i logowaniem.

In [0]:
def auto_cast_columns(df, schema: dict):
    """
    Casts DataFrame columns to specified types if needed.

    :param df: pd.DataFrame
    :param schema: Dict {column_name: type} — e.g. {"age": int, "name": str}
    :return: df with casted columns
    """
    df_copy = df.copy()
    for col, dtype in schema.items():
        if col in df.columns:
            try:
                df_copy[col] = df[col].astype(dtype)
            except Exception as e:
                raise TypeError(f"Failed to cast column '{col}' to {dtype}: {e}")
    return df_copy


## 5. safe_column_rename(df, rename_map)
Cel: Chroni przed błędami podczas zmiany nazw kolumn (np. gdy kolumny nie istnieją).

In [0]:
def safe_column_rename(df, rename_map: dict):
    """
    Safely renames columns if they exist in the DataFrame.

    :param df: pd.DataFrame
    :param rename_map: Dict {old_name: new_name}
    :return: Renamed DataFrame
    """
    missing = [col for col in rename_map.keys() if col not in df.columns]
    if missing:
        raise KeyError(f"Cannot rename missing columns: {missing}")
    return df.rename(columns=rename_map)
